# Multi-follower Stackelberg experiment

Each entry in `scenarios`, `lower_nets`, and `mg_buses` describes the same follower. Every follower receives an independent 24-hour price trajectory when more than one follower is configured. Run the notebook from top to bottom in this order: neural-surrogate solver (NN), zero-order solver (CMA-ES), and residual-correction trust-region algorithm.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandapower.networks as pn
 
from stackelberg.followers.bg import create_cigre_residential_network
from stackelberg.followers.ip import create_industrial_park_network
from stackelberg.network import create_case33bw_20kv
from stackelberg.nn_solver import (
    NNSolverExperimentConfig,
    run_nn_solver_experiment,
)
from stackelberg.trust_region import (
    TrustRegionExperimentConfig,
    run_trust_region_experiment,
)
from stackelberg.zero_order import (
    ZeroOrderExperimentConfig,
    run_zero_order_experiment,
)

## Networks and follower mapping

The order is important: follower `i` uses `scenarios[i]`, `lower_nets[i]`, connects at `mg_buses[i]`, and loads `L2O_<follower_id>.pth`.

In [2]:
T = 24
upper_net = create_case33bw_20kv()

# The three lists use the same follower order.
scenarios = ["SB1", "SB2", "BG1", "BG2", "IP1",]
# SB uses pn.create_cigre_network_lv()
# BG uses create_cigre_residential_network()
lower_nets = [
    pn.create_cigre_network_lv(),
    pn.create_cigre_network_lv(),
    create_cigre_residential_network(),
    create_cigre_residential_network(),
    create_industrial_park_network(),
]
mg_buses = [17, 32, 15, 21, 1]

rng1 = np.random.default_rng(42)
rng2 = np.random.default_rng(1)
# Shared initial prices make the three methods directly comparable.
x0 = np.full((len(scenarios), T), 0.75) # the center point 9.12
# x0 = np.full((len(scenarios), T), 0.0) # the lower bound  9.13
# x0 = np.full((len(scenarios), T), 1.5) # the upper bound 9.13
# x0 = rng1.random((len(scenarios), T)) * 1.5 # the random starting point 9.14
# x0 = rng2.random((len(scenarios), T)) * 1.5 # the random starting point 9.14
x0

array([[0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75],
       [0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75],
       [0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75],
       [0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75],
       [0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75, 0.75,
        0.75, 0.75]])

## 1. Neural-surrogate solver (NN) or saved warm start

Set `RUN_NN_SOLVER=True` to run and save a new NN solution. Leave it `False` to read the last recorded price vector from the newest `results/NN_solver_history_*.csv` without rerunning the solver.

In [3]:
RUN_NN_SOLVER = False

nn_config = NNSolverExperimentConfig(
    upper_net=upper_net,
    lower_nets=lower_nets,
    mg_buses=mg_buses,
    scenarios=scenarios,
    T=T,
    seed=42,
    device="cpu",
    x0=x0,
    steps=10000,
    lr=1e-2,
    save_every=200,
    history_path="results/NN_solver_history.csv",
    evaluate_true=True,
    save_results=True,
)

if RUN_NN_SOLVER:
    nn_result = run_nn_solver_experiment(nn_config)
    nn_prices = np.asarray(nn_result["optimal pi"]).reshape(len(scenarios), T)
    warm_start_point = nn_prices.copy()
    print("Ran NN solver and saved history to:", nn_result.get("history_path"))
else:
    results_dir = Path("results/000")
    history_candidates = list(results_dir.glob("NN_solver_history_*.csv"))
    if not history_candidates:
        fallback_history = results_dir / "NN_solver_history.csv"
        if fallback_history.is_file():
            history_candidates = [fallback_history]
    if not history_candidates:
        raise FileNotFoundError(
            "No NN solver history found in results/. Expected NN_solver_history_*.csv."
        )

    latest_nn_history_path = max(
        history_candidates,
        key=lambda path: path.stat().st_mtime_ns,
    )
    nn_history = pd.read_csv(latest_nn_history_path)
    if nn_history.empty:
        raise ValueError(f"NN solver history is empty: {latest_nn_history_path}")

    pi_columns = sorted(
        (column for column in nn_history.columns if column.startswith("pi_")),
        key=lambda column: int(column.removeprefix("pi_")),
    )
    expected_price_count = len(scenarios) * T
    if len(pi_columns) != expected_price_count:
        raise ValueError(
            f"Expected {expected_price_count} price columns, found {len(pi_columns)} "
            f"in {latest_nn_history_path}."
        )

    last_nn_row = nn_history.sort_values("iteration").iloc[-1]
    nn_prices = last_nn_row[pi_columns].to_numpy(dtype=float).reshape(
        len(scenarios), T
    )
    warm_start_point = nn_prices.copy()
    nn_result = {
        "optimal pi": nn_prices,
        "true_obj": float(last_nn_row["true_loss"])
        if "true_loss" in last_nn_row.index
        else np.nan,
        "runtime_seconds": np.nan,
        "history_path": str(latest_nn_history_path),
    }
    print("Loaded NN warm start from:", latest_nn_history_path)
    print("Saved iteration:", int(last_nn_row["iteration"]))

print("NN true objective:", nn_result["true_obj"])
print("Runtime seconds:", nn_result["runtime_seconds"])
display(pd.DataFrame(nn_prices, index=scenarios))

Loaded NN warm start from: results/000/NN_solver_history_20260912-234954.csv
Saved iteration: 9800
NN true objective: -103.73745796679384
Runtime seconds: nan


,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
SB1,0.905496,0.972395,0.992483,0.998625,0.982330,0.981594,1.006895,1.002889,0.991757,1.000325,...,1.037887,1.018056,1.024587,0.997190,0.993872,1.024512,1.423613,1.006008,1.086199,0.996892
SB2,0.984292,0.985921,0.971553,0.977133,0.962063,0.958349,1.004334,0.994941,1.498616,0.988460,...,1.011563,0.999690,0.996986,0.981880,1.001129,1.013935,0.991371,0.986417,1.089251,0.993309
BG1,1.499927,1.499906,1.499724,1.499684,1.499929,1.499869,1.499946,1.499939,1.499915,1.499948,...,1.499942,1.499932,1.499939,1.499932,1.499938,1.499940,1.499914,0.261827,1.499944,1.499898
BG2,1.499915,1.499862,1.499867,1.499870,1.499940,1.499835,1.499953,1.499933,1.499934,1.499927,...,1.499954,1.499947,1.499947,1.499948,1.499952,1.499953,1.499886,1.499871,1.499922,1.499468
IP1,1.493962,1.488898,1.495653,1.494458,1.490675,1.476853,1.496844,1.499135,1.499072,1.499319,...,1.499453,1.499741,1.499321,1.497359,1.494321,1.498004,1.496904,1.494693,1.498408,1.498219


In [4]:
assert warm_start_point.shape == (len(scenarios), T)
warm_start_point

array([[0.90549648, 0.972395  , 0.99248326, 0.99862492, 0.98232985,
        0.98159385, 1.00689471, 1.00288939, 0.99175656, 1.00032473,
        1.00140524, 0.99738228, 1.01915622, 0.99063975, 1.03788662,
        1.01805568, 1.02458739, 0.99719042, 0.99387169, 1.02451158,
        1.42361331, 1.00600767, 1.0861994 , 0.99689198],
       [0.98429203, 0.98592103, 0.97155261, 0.97713321, 0.96206295,
        0.95834911, 1.00433409, 0.99494135, 1.49861562, 0.98846006,
        0.98933589, 0.99439555, 1.00170684, 0.9589138 , 1.01156259,
        0.99968988, 0.99698627, 0.98187965, 1.00112867, 1.01393485,
        0.99137068, 0.98641741, 1.08925092, 0.99330902],
       [1.49992728, 1.49990582, 1.49972367, 1.49968433, 1.49992895,
        1.49986863, 1.49994588, 1.4999392 , 1.49991465, 1.49994779,
        1.49991512, 1.49993312, 1.499933  , 1.49989676, 1.49994242,
        1.49993229, 1.49993873, 1.49993169, 1.49993753, 1.49994016,
        1.49991381, 0.26182657, 1.49994385, 1.49989808],
       [1.499

## 2. Zero-order solver (CMA-ES) or saved result

Set `RUN_CMA_ES=True` to run and save a new CMA-ES solution. Leave it `False` to load the newest `results/zero_solver_result_*.json` and its matching interaction history.

In [5]:
RUN_CMA_ES = False

zero_config = ZeroOrderExperimentConfig(
    upper_net=upper_net,
    lower_nets=lower_nets,
    mg_buses=mg_buses,
    scenarios=scenarios,
    T=T,
    seed=42,
    pi_0=warm_start_point,
    maxiter=500,
    cma_verbose=1,
    # Per-coordinate initial standard deviation. Use None for the legacy
    # (pi_max - pi_min) / 4 value; 0.05-0.075 favors local warm-start search.
    cma_sigma0= 0.05,
    # Applied to every individual Gurobi lower-level solve.
    lower_time_limit=30.0,
    lower_mip_gap=1e-3,
    save_results=True,
    output_dir="results/000",
)

if RUN_CMA_ES:
    zero_result = run_zero_order_experiment(zero_config)
    print("Ran CMA-ES and saved:", zero_result.get("saved_paths"))
else:
    results_dir = Path(zero_config.output_dir)
    result_candidates = list(results_dir.glob("zero_solver_result_*.json"))
    if not result_candidates:
        raise FileNotFoundError(
            "No saved CMA-ES result found in results/. "
            "Expected zero_solver_result_*.json."
        )

    latest_zero_result_path = max(
        result_candidates,
        key=lambda path: path.stat().st_mtime_ns,
    )
    with latest_zero_result_path.open("r", encoding="utf-8") as file:
        saved_zero_result = json.load(file)

    saved_scenarios = [str(value) for value in saved_zero_result["scenarios"]]
    if saved_scenarios != list(scenarios):
        raise ValueError(
            f"Saved CMA-ES scenarios {saved_scenarios} do not match {scenarios}."
        )
    saved_prices = np.asarray(saved_zero_result["optimal_pi"], dtype=float)
    if saved_prices.size != len(scenarios) * T:
        raise ValueError(
            f"Expected {len(scenarios) * T} saved CMA-ES prices, "
            f"found {saved_prices.size}."
        )

    result_timestamp = latest_zero_result_path.stem.removeprefix(
        "zero_solver_result_"
    )
    zero_history_path = results_dir / f"zero_solver_history_{result_timestamp}.csv"
    zero_solution_path = results_dir / f"zero_solver_solution_{result_timestamp}.csv"
    if not zero_history_path.is_file():
        raise FileNotFoundError(
            f"Matching CMA-ES interaction history not found: {zero_history_path}"
        )

    zero_result = {
        "optimal pi": saved_prices.reshape(len(scenarios), T),
        "optimal value": float(saved_zero_result["optimal_value"]),
        "p_mg": saved_zero_result.get("p_mg"),
        "runtime_seconds": float(saved_zero_result["runtime_seconds"]),
        "n_interactions": int(saved_zero_result["n_interactions"]),
        "interaction_history": pd.read_csv(zero_history_path),
        "scenarios": saved_scenarios,
        "mg_buses": saved_zero_result.get("mg_buses"),
        "saved_paths": {
            "json": str(latest_zero_result_path),
            "solution_csv": str(zero_solution_path),
            "history_csv": str(zero_history_path),
        },
    }
    print("Loaded CMA-ES result from:", latest_zero_result_path)

zero_prices = np.asarray(zero_result["optimal pi"]).reshape(len(scenarios), T)
print("CMA-ES optimal value:", zero_result["optimal value"])
print("Runtime seconds:", zero_result["runtime_seconds"])
print("Composite high-fidelity queries:", zero_result["n_interactions"])
print(
    "Individual follower solves:",
    zero_result["n_interactions"] * len(scenarios),
)
print("Saved paths:", zero_result.get("saved_paths"))
display(pd.DataFrame(zero_prices, index=scenarios))

Loaded CMA-ES result from: results/000/zero_solver_result_20260913-043737.json
CMA-ES optimal value: -105.26827811212056
Runtime seconds: 15016.02490401268
Composite high-fidelity queries: 9000
Individual follower solves: 45000
Saved paths: {'json': 'results/000/zero_solver_result_20260913-043737.json', 'solution_csv': 'results/000/zero_solver_solution_20260913-043737.csv', 'history_csv': 'results/000/zero_solver_history_20260913-043737.csv'}


,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
SB1,0.983199,0.957703,0.867122,0.981245,0.999610,0.978856,1.003714,0.988813,0.938678,1.025007,...,1.061136,0.909787,1.014319,1.000353,0.996259,1.014089,1.494585,0.912330,1.098855,0.941668
SB2,1.000499,0.993827,0.964085,0.980807,0.844366,0.888158,0.891734,0.931081,1.474802,1.059139,...,1.012016,1.002472,1.024808,1.000445,0.999985,1.077473,1.036861,0.999023,1.152307,1.001186
BG1,1.484618,1.412143,1.490132,1.374553,1.361300,1.499850,1.491997,1.427190,1.499981,1.499699,...,1.499999,1.469254,1.481204,1.498366,1.497720,1.499819,1.493936,0.263877,1.464536,1.488468
BG2,1.499930,1.476170,1.484341,1.484295,1.495639,1.499294,1.412486,1.486817,1.488964,1.498635,...,1.478942,1.496993,1.497701,1.469153,1.497792,1.443250,1.451042,1.490410,1.498457,1.406763
IP1,1.498897,1.499888,1.499131,1.499202,1.499495,1.497715,1.499785,1.498868,1.498357,1.499787,...,1.499990,1.499904,1.499919,1.498545,1.498591,1.499615,1.499597,1.498640,1.499993,1.499680


In [6]:
baseline_summary = pd.DataFrame(
    [
        {
            "method": "NN solver",
            "true_objective": nn_result["true_obj"],
            "runtime_seconds": nn_result["runtime_seconds"],
        },
        {
            "method": "CMA-ES",
            "true_objective": zero_result["optimal value"],
            "runtime_seconds": zero_result["runtime_seconds"],
        },
    ]
).set_index("method")
display(baseline_summary)

,true_objective,runtime_seconds
method,,
NN solver,-103.737458,NaN
CMA-ES,-105.268278,15016.024904


## 3. Trust-region algorithm

Run the residual-correction trust-region method last. The first leader solve starts from `x0` and uses `steps_init` on the global surrogate only. After that, each iteration samples true follower responses, fits a local residual, and re-solves with `steps_warm`. Sampled high-fidelity responses are saved as `data/X_{SB,BG,IP}_tr.npy` so they can retrain the NN solver without overwriting the original `X_*.npy` files. For a smoke test, reduce `K_max`, `n_samples`, `steps_init`, and `steps_warm`.

In [ ]:
RESIDUAL_MODEL_TYPE = "rbf"  # Use "linear" for an affine residual.
RBF_KERNEL = "cubic"
RBF_SMOOTHING = 1e-3
RBF_EPSILON = None
RESIDUAL_UPDATE_MODE = "accumulate"  # Use "accumulate" for the previous behavior.

tr_config = TrustRegionExperimentConfig(
    upper_net=upper_net,
    lower_nets=lower_nets,
    mg_buses=mg_buses,
    scenarios=scenarios,
    T=T,
    seed=42,
    device="cpu",
    x0=warm_start_point,
    # Applied to every individual Gurobi lower-level solve.
    lower_time_limit=30.0,
    lower_mip_gap=1e-3,
    # Explicitly enable one hard-DistFlow network-state warm start
    # before every local trust-region leader solve.
    copt_initialize_network_state=True,
    tr_params={
        "R_0": 0.05,
        "R_max": 1.0,
        "gamma_dec": 0.7,
        "gamma_inc": 1.5,
        "eta_1": 0.1,
        "eta_2": 0.75,
    },
    algo_params={
        "n_samples": 60,
        "K_max": 60,
        "sampling_method": "lhs",
        "residual_update_mode": RESIDUAL_UPDATE_MODE,
        "rbf_geometry_condition_max": 1e12,
        "solve_params": {
            "steps_init": 0,
            "steps_warm": 2000,
            "lr": 1e-2,
            "penalty_weights": {
                "eq": 1000.0,
                "ineq": 100.0,
                "soc": 100.0,
                "bound": 10.0,
            },
            "tr_weight": 1e6,
            "verbose": True,
        },
    },
    residual_model_params={
        "model_type": RESIDUAL_MODEL_TYPE,
        "rbf_kernel": RBF_KERNEL,
        "rbf_smoothing": RBF_SMOOTHING,
        "rbf_degree": 1,
        "rbf_scale": "trust_radius/000",
        "rbf_epsilon": RBF_EPSILON,
    },
    save_results=True,
    output_dir="results",
    # False: use local TR samples for fitting, but do not retain or export them.
    save_true_responses=False,
)

tr_result = run_trust_region_experiment(tr_config)
assert tr_result["params"]["algo_params"]["solve_params"][
    "copt_initialize_network_state"
] is True
print("COPT network-state initialization enabled.")
tr_result.get("saved_paths")


## Optional: retrain the NN solver on original + trust-region samples

Trust-region high-fidelity queries are stored only when `save_true_responses=True`. They are kept separately from the original training data:

- original: `data/X_SB.npy`, `data/Y_SB.npy` (and the same pattern for BG, IP)
- trust region: `data/X_SB_tr.npy`, `data/Y_SB_tr.npy` (and the same pattern for BG, IP)

To retrain, first set `save_true_responses=True` in `tr_config` and run the trust-region cell. Then set `RETRAIN_NN_FROM_TR_SAMPLES=True` to concatenate both sets (for example `X_BG.npy` + `X_BG_tr.npy`), train `L2O_{scenario}_tr.pth`, and re-solve with the new maps. The original `L2O_{scenario}.pth` checkpoints are left unchanged. The current experiment leaves both switches `False`.

In [ ]:
from L2Otraining import TrainingConfig, train

RETRAIN_NN_FROM_TR_SAMPLES = False

tr_nn_result = None
if RETRAIN_NN_FROM_TR_SAMPLES:
    for scenario in sorted(set(scenarios)):
        train(
            scenario,
            config=TrainingConfig(scenario=scenario, epochs=500, seed=42),
            data_tag="tr",
            include_original=True,
        )
    tr_nn_config = NNSolverExperimentConfig(
        upper_net=upper_net,
        lower_nets=lower_nets,
        mg_buses=mg_buses,
        scenarios=scenarios,
        T=T,
        seed=42,
        device="cpu",
        x0=x0,
        steps=5000,
        lr=1e-2,
        save_every=200,
        evaluate_true=True,
        save_results=True,
        model_tag="tr",
        history_path="results/NN_solver_tr_history.csv",
    )
    tr_nn_result = run_nn_solver_experiment(tr_nn_config)
    print("Retrained NN true objective:", tr_nn_result["true_obj"])
else:
    saved = (tr_result.get("saved_paths") or {}).get("true_responses", {})
    if saved:
        print("Trust-region samples ready for later NN retraining:")
        for scenario, paths in saved.items():
            print(f"  {scenario}: {paths['X']} , {paths['Y']}")
    else:
        print("No trust-region sample files were saved.")

## Compare results

Inspect all three solutions after the NN solver, CMA-ES, and trust-region cells have completed.

In [ ]:
tr_prices = np.asarray(tr_result["x_opt"]).reshape(len(scenarios), T)
history = pd.DataFrame(tr_result["history"])
last_iteration = tr_result["history"][-1]
tr_objective = (
    last_iteration["F_current"]
    if last_iteration["decision"] == "reject"
    else last_iteration["F_candidate"]
)
tr_sampling_queries = tr_result["n_iter"] * (tr_config.algo_params["n_samples"] + 1)

comparison = pd.DataFrame(
    [
        {
            "method": "NN solver",
            "true_objective": nn_result["true_obj"],
            "runtime_seconds": nn_result["runtime_seconds"],
            "composite_queries": np.nan,
            "individual_follower_solves": np.nan,
        },
        {
            "method": "CMA-ES",
            "true_objective": zero_result["optimal value"],
            "runtime_seconds": zero_result["runtime_seconds"],
            "composite_queries": zero_result["n_interactions"],
            "individual_follower_solves": zero_result["n_interactions"] * len(scenarios),
        },
        {
            "method": "Trust region",
            "true_objective": tr_objective,
            "runtime_seconds": tr_result["elapsed_seconds"],
            "composite_queries": tr_sampling_queries,
            "individual_follower_solves": tr_sampling_queries * len(scenarios),
        },
    ]
).set_index("method")

print("Trust region converged:", tr_result["converged"])
print("Trust region iterations:", tr_result["n_iter"])
print("Saved paths:", tr_result.get("saved_paths"))
display(comparison)
display(pd.concat(
    {
        "NN solver": pd.DataFrame(nn_prices, index=scenarios),
        "CMA-ES": pd.DataFrame(zero_prices, index=scenarios),
        "Trust region": pd.DataFrame(tr_prices, index=scenarios),
    },
    names=["method", "scenario"],
))
display(history[[
    "k", "F_current", "F_candidate", "rho", "decision", "R",
    "max_rbf_condition", "geometry_resample_attempts",
]].tail())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
history.plot(x="k", y=["F_current", "F_candidate"], ax=axes[0], title="Objective")
history.plot(x="k", y="rho", ax=axes[1], title="Trust ratio")
history.plot(x="k", y="R", ax=axes[2], title="Trust-region radius")
plt.tight_layout()

In [ ]:
# Compare methods using composite high-fidelity query accounting.
# CMA-ES n_interactions is the number of composite full-price queries.
zero_plot_df = zero_result["interaction_history"].copy()
zero_plot_df = zero_plot_df.sort_values("interaction").reset_index(drop=True)

tr_plot_df = history.copy()
tr_plot_df["accepted"] = tr_plot_df["decision"].str.startswith("accept")
tr_plot_df["cumulative_composite_queries"] = (
    (tr_plot_df["k"] + 1) * (tr_config.algo_params["n_samples"] + 1)
)

# Use the NN true loss as the query-zero incumbent for both warm-started methods.
# Any worse CMA-ES/TR objective is clipped to this baseline in the plot.
nn_true_loss = float(nn_result["true_obj"])
common_initial_objective = nn_true_loss

zero_plot_df = pd.concat(
    [
        # pd.DataFrame(
        #     {
        #         "interaction": [0],
        #         "upper_objective": [common_initial_objective],
        #     }
        # ),
        zero_plot_df[["interaction", "upper_objective"]],
    ],
    ignore_index=True,
)
# zero_plot_df["best_so_far_loss"] = np.minimum(
#     zero_plot_df["upper_objective"].cummin(),
#     nn_true_loss,
# )
zero_plot_df["best_so_far_loss"] = zero_plot_df["upper_objective"].cummin()

best_accepted = common_initial_objective
tr_best_values = []
for _, row in tr_plot_df.iterrows():
    if row["accepted"]:
        best_accepted = min(best_accepted, float(row["F_candidate"]))
    tr_best_values.append(best_accepted)
# tr_plot_df["best_true_objective"] = np.minimum(
#     tr_best_values,
#     nn_true_loss,
# )
tr_plot_df["best_true_objective"] = tr_best_values

tr_plot_df = pd.concat(
    [
        # pd.DataFrame(
        #     {
        #         "cumulative_composite_queries": [0],
        #         "best_true_objective": [common_initial_objective],
        #     }
        # ),
        tr_plot_df[[
            "cumulative_composite_queries",
            "best_true_objective",
        ]],
    ],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    zero_plot_df["interaction"],
    zero_plot_df["best_so_far_loss"],
    linewidth=2.5,
    label="CMA-ES best-so-far",
)
ax.plot(
    tr_plot_df["cumulative_composite_queries"],
    tr_plot_df["best_true_objective"],
    linewidth=2.5,
    label="Trust region best accepted",
)
ax.axhline(
    nn_true_loss,
    color="tab:purple",
    linestyle="--",
    linewidth=2.5,
    label=f"NN solver true loss = {nn_true_loss:.3f}",
)
ax.scatter(
    [0],
    [common_initial_objective],
    color="black",
    s=70,
    zorder=5,
    label="Shared initial point",
)
x_max = max(
    int(zero_plot_df["interaction"].max()),
    int(tr_plot_df["cumulative_composite_queries"].max()),
)
ax.set_xlim(0, x_max)
ax.set_xlabel("Cumulative composite high-fidelity queries")
ax.set_ylabel("Best upper objective / Loss")
ax.set_title("Composite-query accounting")
ax.grid(alpha=0.35)
ax.legend()

fig.tight_layout()
plt.show()

output_path = Path("plots/Cold_start_query_accounting42.pdf")
output_path.parent.mkdir(parents=True, exist_ok=True)

fig.tight_layout()
fig.savefig(
    output_path,
    format="pdf",
    bbox_inches="tight",
)
plt.show()

print(f"Saved figure to: {output_path.resolve()}")